# Notebook 1: Scraping des matchs et boxscores V3 depuis 2010


In [1]:
from datetime import datetime

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate src/ directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from src.nba_scrapping import *
from src.utils import get_latest_file
from src.config import *
import pandas as pd

In [2]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-11-04 22:09:18.822452


# Saisons ciblées


In [3]:
#seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2011, 2015)]
seasons = ["2015-16","2016-17","2017-18","2018-19","2019-20",]
seasons


['2015-16', '2016-17', '2017-18', '2018-19', '2019-20']

In [4]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# 1. Récupération des matchs


In [6]:
path_games = download_games_for_seasons(seasons, DATA_GAMES_DIR, run_timestamp)


Extraction saison 2015-16
Extraction saison 2016-17
Extraction saison 2017-18
Extraction saison 2018-19
Extraction saison 2019-20


## run once : downloade teams

In [7]:
""" from nba_api.stats.static import teams


# Récupère toutes les équipes NBA
nba_teams = teams.get_teams()
teams_df = pd.DataFrame(nba_teams)

# # Garde les colonnes principales pour la lisibilité
main_cols = ['id', 'full_name', 'abbreviation', 'city', 'state', 'year_founded']
teams_df = teams_df[main_cols]

# # Affiche un aperçu
display(teams_df)



# # Sauvegarde en CSV
# #teams_df.to_csv('nba_teams.csv', index=False)

save_dataframe_to_csv(teams_df, DATA_TEAMS_DIR, prefix='nba_teams_', suffix=run_timestamp)
 """

" from nba_api.stats.static import teams\n\n\n# Récupère toutes les équipes NBA\nnba_teams = teams.get_teams()\nteams_df = pd.DataFrame(nba_teams)\n\n# # Garde les colonnes principales pour la lisibilité\nmain_cols = ['id', 'full_name', 'abbreviation', 'city', 'state', 'year_founded']\nteams_df = teams_df[main_cols]\n\n# # Affiche un aperçu\ndisplay(teams_df)\n\n\n\n# # Sauvegarde en CSV\n# #teams_df.to_csv('nba_teams.csv', index=False)\n\nsave_dataframe_to_csv(teams_df, DATA_TEAMS_DIR, prefix='nba_teams_', suffix=run_timestamp)\n "

# 2. Chargement des GAME_IDs pour scraping boxscore


In [8]:



games_file = path_games #get_latest_file(DATA_GAMES_DIR)


print(f"Using games file: {games_file}")

games_df = pd.read_csv(games_file, dtype={'GAME_ID': str})
games_df


Using games file: /home/ju/Documents/Dev/NBA_Predictor/data/legacy/raw/games/2015-16_2019-20/nba_games_2025-11-04_22-09-18.csv


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22015,1610612739,CLE,Cleveland Cavaliers,0021500002,2015-10-27,CLE @ CHI,L,240,95,...,11,39,50,26,5,7,10,21,-2.0,2015-16
1,22015,1610612741,CHI,Chicago Bulls,0021500002,2015-10-27,CHI vs. CLE,W,240,97,...,7,40,47,13,6,10,13,22,2.0,2015-16
2,22015,1610612737,ATL,Atlanta Hawks,0021500001,2015-10-27,ATL vs. DET,L,239,94,...,7,33,40,22,9,4,15,25,-12.0,2015-16
3,22015,1610612765,DET,Detroit Pistons,0021500001,2015-10-27,DET @ ATL,W,239,106,...,23,36,59,23,5,3,15,15,12.0,2015-16
4,22015,1610612744,GSW,Golden State Warriors,0021500003,2015-10-27,GSW vs. NOP,W,241,111,...,21,35,56,29,8,7,20,29,16.0,2015-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12779,42019,1610612747,LAL,Los Angeles Lakers,0041900404,2020-10-06,LAL @ MIA,W,241,102,...,10,32,42,25,5,4,15,14,6.0,2019-20
12780,42019,1610612748,MIA,Miami Heat,0041900405,2020-10-09,MIA @ LAL,W,241,111,...,9,26,35,26,7,3,13,19,3.0,2019-20
12781,42019,1610612747,LAL,Los Angeles Lakers,0041900405,2020-10-09,LAL vs. MIA,L,240,108,...,12,29,41,21,10,5,15,21,-3.0,2019-20
12782,42019,1610612748,MIA,Miami Heat,0041900406,2020-10-11,MIA vs. LAL,L,240,93,...,9,32,41,25,4,4,13,18,-13.0,2019-20


In [9]:
# import os
# import re

# # 👉 Liste ici les dossiers des saisons à corriger
# season_dirs = [
#     "data/raw/boxscores/batches/2023-24", 
#     #"data/raw/boxscores/batches/2024-25"
# ]

# # ✅ Pattern pour détecter les fichiers batch à corriger
# pattern = re.compile(r"(boxscores_\w+_v3_batch_)(\d+)(\.csv)")

# # 🔁 Pour chaque saison et endpoint
# for season_dir in season_dirs:
#     for endpoint in os.listdir(season_dir):
#         endpoint_path = os.path.join(season_dir, endpoint)
#         if not os.path.isdir(endpoint_path):
#             continue
#         for filename in os.listdir(endpoint_path):
#             match = pattern.match(filename)
#             if match:
#                 prefix, number, suffix = match.groups()
#                 new_number = f"{int(number):03d}"
#                 new_name = f"{prefix}{new_number}{suffix}"
#                 old_path = os.path.join(endpoint_path, filename)
#                 new_path = os.path.join(endpoint_path, new_name)
#                 if old_path != new_path:
#                     os.rename(old_path, new_path)
#                     print(f"✅ Renamed: {filename} ➜ {new_name}")


# 3. Scraping des boxscores V3


In [10]:
#latest batches run_timestamp to complete the boxscores

#latest_batches_run_directory = "2025-06-03_14-23-24"
latest_batches_run_directory = run_timestamp

for season in seasons:
    print(f"--- Traitement de la saison {season} ---")
    season_df = games_df[games_df['SEASON'] == season]
    season_output_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season)
    scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

--- Traitement de la saison 2015-16 ---
[DEBUG] Found 53 batch files for endpoint 'traditional' in season 2015-16
[DEBUG] Found 53 batch files for endpoint 'advanced' in season 2015-16
[DEBUG] Checking last batch file for endpoint 'advanced': /home/ju/Documents/Dev/NBA_Predictor/data/legacy/raw/boxscores/batches/2015-16/advanced/boxscores_advanced_v3_batch_053.csv
[INFO] Resuming from gameId 0041500407 in season 2015-16
[DEBUG] Found 53 batch files for endpoint 'fourfactors' in season 2015-16
[DEBUG] Found 53 batch files for endpoint 'misc' in season 2015-16
[DEBUG] Found 53 batch files for endpoint 'scoring' in season 2015-16
[DEBUG] Found 53 batch files for endpoint 'usage' in season 2015-16
--------- 0 GAME_ID to scrap for season 2015-16 ---------
--- Traitement de la saison 2016-17 ---
[DEBUG] Found 53 batch files for endpoint 'traditional' in season 2016-17
[DEBUG] Found 53 batch files for endpoint 'advanced' in season 2016-17
[DEBUG] Checking last batch file for endpoint 'advance

In [11]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-11-05 05:36:50.448606
Total time:  7:27:31.626154


In [12]:
print("\n✅ Scraping historique V3 terminé")



✅ Scraping historique V3 terminé
